Load the transformed data

Load the transformed data

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

Start Spark

In [2]:
spark = (
    SparkSession.builder
    .appName("SupportIQ-SparkSQL")
    .master("local[*]")
    .getOrCreate()
)

c:\Users\91840\OneDrive\Documents\PYTHON PROJECTS\support-iq\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Define the transformed-data path

In [3]:
transformed_path = "../data/processed/support_tickets_transformed"

Load the Parquet files

In [4]:
parquet_files = [

    os.path.join(
        transformed_path,
        file
    )

    for file in os.listdir(
        transformed_path
    )

    if file.endswith(".parquet")

]

In [5]:
print(
    "Number of Parquet files:",
    len(parquet_files)
)

Number of Parquet files: 8


In [6]:
tickets_df = spark.read.parquet(
    *parquet_files
)

print(
    "Transformed data loaded successfully."
)

Transformed data loaded successfully.


Create a Temporary SQL View

In [7]:
tickets_df.createOrReplaceTempView(
    "support_tickets"
)

Run your first Spark SQL query

In [8]:
result_df = spark.sql(
    """
    SELECT *
    FROM support_tickets
    LIMIT 10
    """
)

In [9]:
result_df.show(
    truncate=False
)

+----------+-----------+--------+------------+-----------+--------+---------------+-----------------+-------+-----------+-------+--------------------------------------------------+----------------------------------------------------+---------------+----------------------+----------------+---------------------+---------------+------------+-------------+-------------------+-----------------------+---------------------+--------------------+--------------+
|ticket_id |customer_id|agent_id|created_date|closed_date|priority|category       |sub_category     |product|status     |channel|issue_description                                 |resolution                                          |customer_region|first_response_minutes|resolution_hours|customer_satisfaction|escalation_flag|created_year|created_month|resolution_category|first_response_category|satisfaction_category|escalation_indicator|priority_level|
+----------+-----------+--------+------------+-----------+--------+---------------+---

In [10]:
tickets_df.show(10)

+----------+-----------+--------+------------+-----------+--------+---------------+-----------------+-------+-----------+-------+--------------------+--------------------+---------------+----------------------+----------------+---------------------+---------------+------------+-------------+-------------------+-----------------------+---------------------+--------------------+--------------+
| ticket_id|customer_id|agent_id|created_date|closed_date|priority|       category|     sub_category|product|     status|channel|   issue_description|          resolution|customer_region|first_response_minutes|resolution_hours|customer_satisfaction|escalation_flag|created_year|created_month|resolution_category|first_response_category|satisfaction_category|escalation_indicator|priority_level|
+----------+-----------+--------+------------+-----------+--------+---------------+-----------------+-------+-----------+-------+--------------------+--------------------+---------------+----------------------+

Select specific columns

In [11]:
result_df = spark.sql(
    """
    SELECT
        ticket_id,
        customer_id,
        priority,
        category,
        status,
        channel,
        resolution_hours,
        customer_satisfaction
    FROM support_tickets
    LIMIT 20
    """
)

result_df.show(
    truncate=False
)

+----------+-----------+--------+---------------+-----------+-------+----------------+---------------------+
|ticket_id |customer_id|priority|category       |status     |channel|resolution_hours|customer_satisfaction|
+----------+-----------+--------+---------------+-----------+-------+----------------+---------------------+
|TKT0000127|CUST48563  |Low     |Payroll        |In Progress|Chat   |147.42          |1.0                  |
|TKT0000173|CUST18535  |Low     |Core HR        |Closed     |Phone  |49.02           |4.0                  |
|TKT0000764|CUST01754  |Medium  |Benefits       |In Progress|Chat   |5.38            |1.0                  |
|TKT0001871|CUST18157  |Critical|Performance    |Open       |Email  |170.18          |3.0                  |
|TKT0002017|CUST20223  |Medium  |Time Management|Open       |Phone  |125.28          |5.0                  |
|TKT0002379|CUST47850  |Critical|Recruitment    |In Progress|Phone  |27.61           |2.0                  |
|TKT0002641|CUST026

Count total tickets

In [12]:
result_df = spark.sql(
    """
    SELECT
        COUNT(*) AS total_tickets
    FROM support_tickets
    """
)

result_df.show()

+-------------+
|total_tickets|
+-------------+
|      1204092|
+-------------+



Count tickets by priority

In [13]:
result_df = spark.sql(
    """
    SELECT
        priority,
        COUNT(*) AS ticket_count
    FROM support_tickets
    GROUP BY priority
    ORDER BY ticket_count DESC
    """
)

result_df.show()

+--------+------------+
|priority|ticket_count|
+--------+------------+
|    High|      289795|
|Critical|      280849|
|  Medium|      276597|
|     Low|      275946|
| Unknown|       80905|
+--------+------------+



Count tickets by category

In [14]:
result_df = spark.sql(
    """
    SELECT
        category,
        COUNT(*) AS ticket_count
    FROM support_tickets
    GROUP BY category
    ORDER BY ticket_count DESC
    """
)

result_df.show()

+---------------+------------+
|       category|ticket_count|
+---------------+------------+
|        Payroll|      201174|
|    Recruitment|      200888|
|       Benefits|      200737|
|        Core HR|      200461|
|Time Management|      200438|
|    Performance|      200394|
+---------------+------------+



Average resolution time by category

In [15]:
result_df = spark.sql(
    """
    SELECT
        category,
        ROUND(
            AVG(resolution_hours),
            2
        ) AS average_resolution_hours
    FROM support_tickets
    GROUP BY category
    ORDER BY average_resolution_hours DESC
    """
)

result_df.show()

+---------------+------------------------+
|       category|average_resolution_hours|
+---------------+------------------------+
|Time Management|                  100.53|
|        Payroll|                  100.49|
|    Recruitment|                  100.45|
|    Performance|                  100.42|
|        Core HR|                  100.41|
|       Benefits|                  100.39|
+---------------+------------------------+



Average customer satisfaction by category

In [16]:
result_df = spark.sql(
    """
    SELECT
        category,
        ROUND(
            AVG(customer_satisfaction),
            2
        ) AS average_satisfaction
    FROM support_tickets
    GROUP BY category
    ORDER BY average_satisfaction DESC
    """
)

result_df.show()

+---------------+--------------------+
|       category|average_satisfaction|
+---------------+--------------------+
|        Payroll|                 3.0|
|        Core HR|                 3.0|
|    Recruitment|                 3.0|
|Time Management|                 3.0|
|    Performance|                 3.0|
|       Benefits|                 3.0|
+---------------+--------------------+



Tickets by channel

In [17]:
result_df = spark.sql(
    """
    SELECT
        channel,
        COUNT(*) AS ticket_count
    FROM support_tickets
    GROUP BY channel
    ORDER BY ticket_count DESC
    """
)

result_df.show()

+-------+------------+
|channel|ticket_count|
+-------+------------+
|  Phone|      301532|
|  Email|      301326|
|   Chat|      300994|
| Portal|      300240|
+-------+------------+



Tickets by status

In [18]:
result_df = spark.sql(
    """
    SELECT
        status,
        COUNT(*) AS ticket_count
    FROM support_tickets
    GROUP BY status
    ORDER BY ticket_count DESC
    """
)

result_df.show()

+-----------+------------+
|     status|ticket_count|
+-----------+------------+
|In Progress|      301697|
|     Closed|      301277|
|       Open|      300816|
|   Resolved|      300302|
+-----------+------------+



Escalation analysis

In [19]:
result_df = spark.sql(
    """
    SELECT
        SUM(escalation_indicator) AS total_escalated_tickets
    FROM support_tickets
    """
)

result_df.show()

+-----------------------+
|total_escalated_tickets|
+-----------------------+
|                 601036|
+-----------------------+



Escalation rate

In [20]:
result_df = spark.sql(
    """
    SELECT
        ROUND(
            SUM(escalation_indicator)
            / COUNT(*) * 100,
            2
        ) AS escalation_rate_percentage
    FROM support_tickets
    """
)

result_df.show()

+--------------------------+
|escalation_rate_percentage|
+--------------------------+
|                     49.92|
+--------------------------+



Average resolution time by priority

In [21]:
result_df = spark.sql(
    """
    SELECT
        priority,
        COUNT(*) AS ticket_count,
        ROUND(
            AVG(resolution_hours),
            2
        ) AS average_resolution_hours
    FROM support_tickets
    GROUP BY priority
    ORDER BY average_resolution_hours DESC
    """
)

result_df.show()

+--------+------------+------------------------+
|priority|ticket_count|average_resolution_hours|
+--------+------------+------------------------+
| Unknown|       80905|                  100.62|
|    High|      289795|                  100.53|
|     Low|      275946|                  100.52|
|  Medium|      276597|                  100.36|
|Critical|      280849|                  100.33|
+--------+------------+------------------------+



Monthly ticket volume

In [22]:
result_df = spark.sql(
    """
    SELECT
        created_year,
        created_month,
        COUNT(*) AS ticket_count
    FROM support_tickets
    GROUP BY
        created_year,
        created_month
    ORDER BY
        created_year,
        created_month
    """
)

result_df.show()

+------------+-------------+------------+
|created_year|created_month|ticket_count|
+------------+-------------+------------+
|        NULL|         NULL|       12240|
|        2025|            1|       60765|
|        2025|            2|       54871|
|        2025|            3|       60694|
|        2025|            4|       58838|
|        2025|            5|       61023|
|        2025|            6|       58682|
|        2025|            7|       60543|
|        2025|            8|       60519|
|        2025|            9|       58985|
|        2025|           10|       60448|
|        2025|           11|       58647|
|        2025|           12|       60866|
|        2026|            1|       60696|
|        2026|            2|       54508|
|        2026|            3|       60978|
|        2026|            4|       58515|
|        2026|            5|       60434|
|        2026|            6|       58919|
|        2026|            7|       60327|
+------------+-------------+------

Find high-priority tickets

In [23]:
result_df = spark.sql(
    """
    SELECT
        ticket_id,
        priority,
        category,
        status,
        resolution_hours
    FROM support_tickets
    WHERE priority IN ('High', 'Critical')
    LIMIT 20
    """
)

result_df.show(
    truncate=False
)

+----------+--------+---------------+-----------+----------------+
|ticket_id |priority|category       |status     |resolution_hours|
+----------+--------+---------------+-----------+----------------+
|TKT0001871|Critical|Performance    |Open       |170.18          |
|TKT0002379|Critical|Recruitment    |In Progress|27.61           |
|TKT0002641|High    |Recruitment    |Closed     |94.73           |
|TKT0002874|Critical|Time Management|In Progress|107.21          |
|TKT0002985|Critical|Recruitment    |Resolved   |100.02          |
|TKT0003396|High    |Core HR        |Resolved   |190.28          |
|TKT0003444|High    |Recruitment    |Resolved   |15.92           |
|TKT0005263|Critical|Core HR        |Closed     |126.16          |
|TKT0005586|High    |Time Management|Resolved   |103.89          |
|TKT0005983|Critical|Payroll        |Open       |40.39           |
|TKT0006712|High    |Core HR        |In Progress|59.24           |
|TKT0007702|Critical|Time Management|Closed     |28.93        

Find slow-resolution tickets

In [24]:
result_df = spark.sql(
    """
    SELECT
        ticket_id,
        category,
        priority,
        resolution_hours
    FROM support_tickets
    WHERE resolution_hours > 72
    ORDER BY resolution_hours DESC
    LIMIT 20
    """
)

result_df.show(
    truncate=False
)

+----------+---------------+--------+----------------+
|ticket_id |category       |priority|resolution_hours|
+----------+---------------+--------+----------------+
|TKT0508406|Recruitment    |Critical|200.0           |
|TKT0060681|Benefits       |Low     |200.0           |
|TKT0415222|Recruitment    |Critical|200.0           |
|TKT0197853|Benefits       |Medium  |200.0           |
|TKT0559568|Payroll        |Critical|200.0           |
|TKT0800765|Performance    |Critical|200.0           |
|TKT0992282|Performance    |Critical|200.0           |
|TKT0019922|Time Management|Unknown |200.0           |
|TKT0264453|Payroll        |High    |200.0           |
|TKT0273931|Recruitment    |Critical|200.0           |
|TKT0365598|Performance    |Low     |200.0           |
|TKT0460212|Time Management|High    |200.0           |
|TKT0799889|Core HR        |High    |200.0           |
|TKT0276248|Benefits       |Critical|200.0           |
|TKT0589580|Recruitment    |High    |200.0           |
|TKT062947

Combine conditions

In [25]:
result_df = spark.sql(
    """
    SELECT
        ticket_id,
        priority,
        category,
        resolution_hours,
        status
    FROM support_tickets
    WHERE priority = 'Critical'
      AND resolution_hours > 48
    ORDER BY resolution_hours DESC
    """
)

result_df.show(
    20,
    truncate=False
)

+----------+--------+-----------+----------------+-----------+
|ticket_id |priority|category   |resolution_hours|status     |
+----------+--------+-----------+----------------+-----------+
|TKT0603137|Critical|Core HR    |200.0           |Closed     |
|TKT0273931|Critical|Recruitment|200.0           |In Progress|
|TKT0276248|Critical|Benefits   |200.0           |Open       |
|TKT0629472|Critical|Payroll    |200.0           |In Progress|
|TKT1164120|Critical|Core HR    |200.0           |Open       |
|TKT0559568|Critical|Payroll    |200.0           |Closed     |
|TKT0800765|Critical|Performance|200.0           |In Progress|
|TKT0992282|Critical|Performance|200.0           |Resolved   |
|TKT0197346|Critical|Recruitment|200.0           |Resolved   |
|TKT0415222|Critical|Recruitment|200.0           |Closed     |
|TKT0508406|Critical|Recruitment|200.0           |Resolved   |
|TKT1007211|Critical|Recruitment|199.99          |Closed     |
|TKT1094889|Critical|Benefits   |199.99          |Open 

Joins

In [40]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

Start Spark

In [41]:
spark = (
    SparkSession.builder
    .appName("SupportIQ-Joins")
    .master("local[*]")
    .getOrCreate()
)

Define the path

In [42]:
transformed_path = "../data/processed/support_tickets_transformed"

Create a customer DataFrame

In [43]:
customer_data = [
    ("CUST00001", "Asia Pacific", "Enterprise"),
    ("CUST00002", "Europe", "Standard"),
    ("CUST00003", "North America", "Enterprise"),
    ("CUST00004", "Latin America", "Standard"),
    ("CUST00005", "Europe", "Enterprise")
]

In [45]:
customer_df = spark.createDataFrame(
    customer_data,
    [
        "customer_id",
        "customer_region",
        "customer_type"
    ]
)

In [46]:
customer_df.show()

Py4JJavaError: An error occurred while calling o228.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 46.0 failed 1 times, most recent failure: Lost task 0.0 in stage 46.0 (TID 126) (192.168.1.4 executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:303)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:154)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:182)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:330)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:73)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:374)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:338)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:57)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoi

INNER JOIN